In [1]:
import sys
sys.path.append("..")
import pandas as pd

import matplotlib.pyplot as plt
import statsmodels.api as sm

from src.data import load_brent_spot, load_brent_futures
from src.models import random_walk_forecast, ar1_forecast, historical_mean_forecast
from src.features import log_returns, realised_volatility
from src.backtest import walk_forward
from src.metrics import compare_models

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [8]:
spot = load_brent_spot(start="1987-01-01")
data = spot.copy()
data["log_return"] = log_returns(data, col="brent_spot")
data = data.dropna()


test_args = dict(horizon=60, start_date="2024-09-30", end_date="2026-09-30", target_end_date="2026-09-30")


test_results = {
    "Random Walk": walk_forward(data, forecast_funct=random_walk_forecast, **test_args),
    "Historical Mean": walk_forward(data, forecast_funct=historical_mean_forecast, lookback=3000, **test_args),
    "AR": walk_forward(data, forecast_funct=ar_forecast, lookback=3000, **test_args),
    "Direct forecast": walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **test_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **test_args),
    "Direct lag_5 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=5, long=60, lookback=3000, **test_args),
}

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0


### Mean Revert

In [9]:
# PRICES = ["brent_spot", "wti", "usd", "sp500"]   # prices -> log returns
# INDICES = ["vix", "ovx", "epu", "gpr"]           # fear/uncertainty indices -> log levels

df_mr = load_dataset()
df_mr = df_mr[["brent_spot", "wti"]]
df_mr = df_mr.rename(columns={"wti": "wti_spot"})
df_mr["log_return"] = np.log(df_mr["brent_spot"]).diff()
df_mr = df_mr.dropna()
df_mr.head()

,brent_spot,wti_spot,log_return
DATE,,,
1987-05-21,18.45,19.95,-0.009709
1987-05-22,18.55,19.68,0.005405
1987-05-25,18.60,19.68,0.002692
1987-05-26,18.63,19.35,0.001612
1987-05-27,18.60,19.38,-0.001612


In [10]:
res = walk_forward(df_mr, forecast_funct=spread_mr_forecast, lookback=3000, **test_args)
res.head()

,origin_date,target_date,origin_price,forecast,actual
0,2024-09-30,2024-12-23,72.35,72.821259,72.12
1,2024-10-01,2024-12-24,75.30,74.677758,73.50
2,2024-10-02,2024-12-27,74.86,74.955398,73.77
3,2024-10-03,2024-12-30,77.57,78.682642,74.24
4,2024-10-04,2024-12-31,79.32,79.386758,74.58


In [11]:
test_results["Mean Revert"] = res

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0


### XGBoost

In [12]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

def _make_features(r):
    s = pd.Series(r)
    F = pd.DataFrame({f"lag{k}": s.shift(k) for k in range(5)})  # last 5 daily returns
    for w in (5, 20, 60):
        F[f"ret{w}"] = s.rolling(w).sum()    # momentum over several horizons
    for w in (20, 60):
        F[f"vol{w}"] = s.rolling(w).std()    # volatility regime
    return F

def xgb_direct_forecast(df, origin_idx, horizon=60, fit_idx=None, lookback=3000,
                        val_frac=0.2, seed=0):
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"
    start = fit_idx - lookback + 1
    assert start >= 0, "not enough history for the lookback window"

    r = df["log_return"].to_numpy(dtype=float)[: origin_idx + 1]
    cs = np.cumsum(np.nan_to_num(r))
    F = _make_features(r)

    # Sample at time t: features known at t, target = sum of r[t+1 .. t+horizon]
    t_all = np.arange(max(start, 60), fit_idx - horizon + 1)   # 60 = rolling warm-up
    X_all = F.iloc[t_all].to_numpy()
    y_all = cs[t_all + horizon] - cs[t_all]
    n = len(t_all)

    # Purged chronological split: gap of `horizon` between train and validation
    n_val = int(n * val_frac)
    n_tr = n - n_val - horizon
    assert n_tr > 250, "too few samples; increase lookback"
    tr, va = np.arange(n_tr), np.arange(n - n_val, n)

    params = dict(learning_rate=0.02, max_depth=2, min_child_weight=20,
                  subsample=0.7, colsample_bytree=0.7, reg_lambda=10.0,
                  random_state=seed, n_jobs=-1)

    # Step 1: early stopping on the validation block picks the number of trees
    m = XGBRegressor(n_estimators=1500, early_stopping_rounds=50, **params)
    m.fit(X_all[tr], y_all[tr], eval_set=[(X_all[va], y_all[va])], verbose=False)
    best_n = m.best_iteration + 1
    val_rmse = np.sqrt(np.mean((y_all[va] - m.predict(X_all[va])) ** 2))
    rw_rmse = np.sqrt(np.mean(y_all[va] ** 2))                 # zero-return benchmark

    # Step 2: refit on all samples with that number of trees
    final = XGBRegressor(n_estimators=best_n, **params).fit(X_all, y_all)

    x0 = F.iloc[[origin_idx]]
    assert not x0.isna().any().any(), "NaNs in origin features"
    pred_ret = float(final.predict(x0)[0])

    p_t = df["brent_spot"].iloc[origin_idx]
    
    return p_t * np.exp(pred_ret)

In [13]:
res = walk_forward(data, forecast_funct=xgb_direct_forecast, lookback=3000, **test_args)
res.head()

,origin_date,target_date,origin_price,forecast,actual
0,2024-09-30,2024-12-23,72.35,61.784337,72.12
1,2024-10-01,2024-12-24,75.30,69.099503,73.50
2,2024-10-02,2024-12-27,74.86,69.462503,73.77
3,2024-10-03,2024-12-30,77.57,77.202936,74.24
4,2024-10-04,2024-12-31,79.32,79.568870,74.58


In [14]:
test_results["XGBoost"] = res

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0
XGBoost,16.214299,24.678568,18.584187,0.534676,0.506135,0.778302,0.613383,0.611570,0.314894,0.415730,0.085776,0.285877,-0.246929,7.0


### Direct Ridge Forecast

In [15]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

def make_features(df, lvl_cols, windows):
    lb = np.log(df["brent_spot"])
    F = {"spread": (df["brent_spot"] - df["wti_spot"]) / df["brent_spot"]}   # WTI-safe
    for w in windows:
        F[f"brent_{w}d"] = lb.diff(w)                      # Brent trailing log return
        for c in lvl_cols:
            F[f"{c}_{w}d"] = df[c].diff(w)                 # change in log level
    return pd.DataFrame(F)

def direct_ridge_forecast(df, origin_idx, horizon=60, fit_idx=None, lookback=3000,
                          lvl_cols=("usd", "sp500", "ovx", "vix"), windows=(20, 60),
                          alphas=np.logspace(1, 5, 9)):
    
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"
    start = fit_idx - lookback + 1
    assert start >= 0, "not enough history for the lookback window"

    lb = np.log(df["brent_spot"])
    F = make_features(df, lvl_cols, windows)
    y = lb.shift(-horizon) - lb                            # next-60-day Brent log return

    # Features known at t; target must be known by fit_idx
    t = np.arange(start + max(windows), fit_idx - horizon + 1)
    assert len(t) > 4 * horizon, "too few samples; increase lookback"
    X, yt = F.iloc[t].to_numpy(), y.iloc[t].to_numpy()
    assert not (np.isnan(X).any() or np.isnan(yt).any()), "NaNs in fit window"
    x0 = F.iloc[[origin_idx]].to_numpy()
    assert not np.isnan(x0).any(), "NaNs in origin features"

    # Scale -> clip at +/-4 sd -> ridge, all refitted inside each CV fold (no leakage)
    pipe = make_pipeline(StandardScaler(),
                         FunctionTransformer(lambda Z: np.clip(Z, -4, 4)),
                         Ridge(fit_intercept=False))
    cv = TimeSeriesSplit(n_splits=5, gap=horizon)          # gap purges overlapping targets
    gs = GridSearchCV(pipe, {"ridge__alpha": alphas}, cv=cv,
                      scoring="neg_root_mean_squared_error").fit(X, yt)

    cv_rmse = -gs.best_score_
    rw_rmse = np.mean([np.sqrt(np.mean(yt[te] ** 2)) for _, te in cv.split(X)])

    pred_ret = float(gs.predict(x0)[0])
    p_t = df["brent_spot"].iloc[origin_idx]
    
    return p_t * np.exp(pred_ret)

In [16]:
df_rr = load_dataset()
  
df_rr = df_rr[["brent_spot", "wti", "usd", "sp500", "vix", "ovx"]]
df_rr = df_rr.rename(columns={"wti": "wti_spot"})
df_rr = df_rr.dropna()

for col in ["usd", "sp500", "vix", "ovx"]:
    df_rr[col] = np.log(df_rr[col])

df_rr.head()

,brent_spot,wti_spot,usd,sp500,vix,ovx
DATE,,,,,,
2007-05-10,64.63,61.85,4.553357,7.307517,2.610070,3.299165
2007-05-11,65.18,62.35,4.553914,7.317113,2.561096,3.273743
2007-05-14,65.86,62.55,4.552051,7.315318,2.636196,3.304319
2007-05-15,66.74,63.16,4.549207,7.314013,2.639771,3.328268
2007-05-16,66.83,62.57,4.551920,7.322603,2.602690,3.298426


In [17]:
res = walk_forward(df_rr, forecast_funct=direct_ridge_forecast, lookback=3000, **test_args)
res.head()

,origin_date,target_date,origin_price,forecast,actual
0,2024-09-30,2024-12-23,72.35,72.690840,72.12
1,2024-10-01,2024-12-24,75.30,75.699659,73.50
2,2024-10-02,2024-12-27,74.86,75.284314,73.77
3,2024-10-03,2024-12-30,77.57,78.054674,74.24
4,2024-10-04,2024-12-31,79.32,79.726752,74.58


In [18]:
test_results["Direct ridge"] = res

compare_models(test_results,  horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0
XGBoost,16.214299,24.678568,18.584187,0.534676,0.506135,0.778302,0.613383,0.611570,0.314894,0.415730,0.085776,0.285877,-0.246929,7.0
Direct ridge,14.265363,22.079316,15.857825,0.689038,0.665158,0.693396,0.678984,0.712389,0.685106,0.698482,0.266544,0.989700,-0.142001,11.0


### Combination Forecast

In [21]:
# =====================================================================
# 1) COMBINATION FORECAST  (average of member forecasts in log-return space)
# =====================================================================
import numpy as np, pandas as pd
from functools import partial
from scipy import stats

def combination_forecast(df, origin_idx, horizon=60, fit_idx=None, members=None, weights=None):
    """Weighted average of member forecasts.
    Members are existing forecast functions with signature (df, origin_idx, horizon, fit_idx).
    Averaging is done on the predicted 60-day LOG return, then converted back to a price,
    so the combination is a geometric mean of the member price forecasts."""
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"

    if members is None:   # choose members BEFORE looking at results; keep RW in as a shrinkage anchor
        members = [
            random_walk_forecast,
            partial(direct_lags_forecast, short=1, long=60, lookback=3000),
            partial(direct_lags_forecast, short=5, long=60, lookback=3000),
            partial(direct_forecast, lookback=3000),
        ]
    w = np.full(len(members), 1 / len(members)) if weights is None else np.asarray(weights, float)
    assert len(w) == len(members) and np.isclose(w.sum(), 1), "weights must match members and sum to 1"

    p_t = df["brent_spot"].iloc[origin_idx]
    rets = np.array([np.log(fn(df, origin_idx, horizon=horizon, fit_idx=fit_idx) / p_t)
                     for fn in members])
    return p_t * np.exp(w @ rets)

### GARCH-based prediction interval

In [22]:
# =====================================================================
# 2) GARCH-BASED PREDICTION INTERVAL
# =====================================================================
from arch import arch_model

def garch_interval_forecast(df, origin_idx, horizon=60, fit_idx=None, lookback=3000,
                            center_fn=None, level=0.95, vol="garch", gjr=False):
    """Prediction interval for Brent at origin+horizon.
    - centre : log-return forecast of `center_fn` (default: random walk, 0 return)
    - width  : sqrt of the summed h-step GARCH(1,1)-t variance forecasts
               (vol="constant" gives the benchmark sd*sqrt(h) from the fit window)
    - params are estimated on data up to fit_idx only; the volatility state is then
      filtered up to origin_idx with params frozen (no look-ahead).
    Returns dict(forecast, lower, upper, sigma_h)."""
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"
    start = fit_idx - lookback + 1
    assert start >= 0, "not enough history for the lookback window"

    r = df["log_return"].to_numpy(float)
    assert not np.isnan(r[start: origin_idx + 1]).any(), "NaNs in return window"
    p_t = df["brent_spot"].iloc[origin_idx]

    used = vol
    if vol == "garch":
        spec = dict(mean="Zero", vol="GARCH", p=1, o=int(gjr), q=1, dist="t", rescale=False)
        fit = arch_model(r[start: fit_idx + 1] * 100, **spec).fit(disp="off")   # pct scale
        if fit.convergence_flag != 0:
            used = "constant (garch failed)"
        else:
            filt = arch_model(r[start: origin_idx + 1] * 100, **spec).fix(fit.params)
            var_path = filt.forecast(horizon=horizon, reindex=False).variance.to_numpy()[-1]
            sigma_h = np.sqrt(var_path.sum()) / 100          # sd of the h-day cumulative log return
    if used != "garch":
        sigma_h = r[start: fit_idx + 1].std(ddof=1) * np.sqrt(horizon)

    pred_ret = 0.0 if center_fn is None else np.log(center_fn(df, origin_idx, horizon, fit_idx) / p_t)
    z = stats.norm.ppf(0.5 + level / 2)    # 60-day sum of returns is close to normal even if daily is fat-tailed
    return {"forecast": p_t * np.exp(pred_ret),
            "lower": p_t * np.exp(pred_ret - z * sigma_h),
            "upper": p_t * np.exp(pred_ret + z * sigma_h),
            "sigma_h": sigma_h, "vol_used": used}


def walk_forward_interval(df, horizon, start_date, end_date, interval_fn,
                          target_end_date=None, **kwargs):
    """Same loop as walk_forward but records lower/upper too (result also works with the metric functions)."""
    dates = df.index
    start_idx, end_idx = dates.searchsorted(start_date), dates.searchsorted(end_date)
    rows = []
    for i in range(start_idx, end_idx):
        j = i + horizon
        if j >= len(dates): break
        if target_end_date is not None and dates[j] > pd.Timestamp(target_end_date): break
        out = interval_fn(df, i, horizon, i, **kwargs)
        rows.append({"origin_date": dates[i], "target_date": dates[j],
                     "origin_price": df["brent_spot"].iloc[i], **out,
                     "actual": df["brent_spot"].iloc[j]})
    return pd.DataFrame(rows)


def interval_metrics(res, level=0.95):
    """Coverage should be close to `level`. Interval (Winkler) score: lower = better (rewards narrow AND correct)."""
    a = 1 - level
    inside = (res["actual"] >= res["lower"]) & (res["actual"] <= res["upper"])
    width = res["upper"] - res["lower"]
    score = width + (2 / a) * (res["lower"] - res["actual"]).clip(lower=0) \
                  + (2 / a) * (res["actual"] - res["upper"]).clip(lower=0)
    return {"coverage": inside.mean(), "target": level, "mean_width": width.mean(),
            "mean_rel_width": (width / res["origin_price"]).mean(), "interval_score": score.mean(),
            "n": len(res), "eff_n (n/h)": len(res) / 60}   # overlapping windows: effective n is ~n/h


### ==== Diebold-Mariano test ====

In [24]:
# =====================================================================
# 3) DIEBOLD-MARIANO TEST (fixed) + relative improvement table
# =====================================================================
def dm_test(results_a, results_b, horizon=60, loss="se"):
    """H0: equal predictive accuracy. d = loss_A - loss_B, so a NEGATIVE statistic means A is better.
    - aligns on (origin_date, target_date), so models are compared on identical forecasts
    - long-run variance: Newey-West with Bartlett weights up to lag h-1 (always >= 0)
    - Harvey-Leybourne-Newbold small-sample correction + Student-t p-values
    loss: 'se' (RMSE), 'ae' (MAE), 'ape' (MAPE)"""
    key = ["origin_date", "target_date"]
    m = (results_a[key + ["actual", "forecast"]]
         .merge(results_b[key + ["forecast"]], on=key, suffixes=("_a", "_b"))
         .sort_values("origin_date"))
    n, h = len(m), horizon
    ea, eb = m["actual"] - m["forecast_a"], m["actual"] - m["forecast_b"]
    f = {"se": lambda e: e**2, "ae": lambda e: e.abs(), "ape": lambda e: e.abs() / m["actual"]}[loss]
    d = (f(ea) - f(eb)).to_numpy()

    out = {"n": n, "MAE_a": ea.abs().mean(), "MAE_b": eb.abs().mean(),
           "RMSE_a": np.sqrt((ea**2).mean()), "RMSE_b": np.sqrt((eb**2).mean()),
           "DM_stat": np.nan, "p_two_sided": np.nan, "p_A_better": np.nan}
    corr = (n + 1 - 2 * h + h * (h - 1) / n) / n
    if n < 3 or corr <= 0:
        out["note"] = f"n={n} too small for h={h} (need roughly n > 2h, ideally n >> h)"
        return out

    dc = d - d.mean()
    L = min(h - 1, n - 1)
    lrv = dc @ dc / n + 2 * sum((1 - k / (L + 1)) * (dc[k:] @ dc[:-k]) / n for k in range(1, L + 1))
    if lrv <= 0 or d.std() == 0:
        out["note"] = "non-positive long-run variance"
        return out
    dm = d.mean() / np.sqrt(lrv / n) * np.sqrt(corr)
    out.update(DM_stat=dm, p_two_sided=2 * stats.t.sf(abs(dm), n - 1), p_A_better=stats.t.cdf(dm, n - 1))
    if n < 5 * h:
        out["note"] = f"n={n} < 5h: p-values unreliable (few independent windows)"
    return out


def dm_table(results_dict, benchmark="Random Walk", horizon=60, loss="se"):
    """Each model vs benchmark: DM test + % improvement in MAE and RMSE (positive = better than benchmark)."""
    rows = {}
    for name, res in results_dict.items():
        if name == benchmark: continue
        t = dm_test(res, results_dict[benchmark], horizon=horizon, loss=loss)
        rows[name] = {"n": t["n"],
                      "MAE": t["MAE_a"], "RMSE": t["RMSE_a"],
                      "MAE_impr_%": 100 * (1 - t["MAE_a"] / t["MAE_b"]),
                      "RMSE_impr_%": 100 * (1 - t["RMSE_a"] / t["RMSE_b"]),
                      "DM_stat": t["DM_stat"], "p_two_sided": t["p_two_sided"],
                      "p_better_than_bm": t["p_A_better"], "note": t.get("note", "")}
    return pd.DataFrame(rows).T


# =====================================================================
# 4) HOW TO RUN  (put in a separate notebook cell, after your existing cells)
# =====================================================================
# Needs a LONG evaluation window: DM needs n >> 60. Fix the design before looking at results.
long_args = dict(horizon=60, start_date="2024-09-30", end_date="2026-09-30", target_end_date="2026-09-30")

results = {
    "Random Walk":         walk_forward(data, forecast_funct=random_walk_forecast, **long_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **long_args),
    "Direct forecast":     walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **long_args),
    "Combination":         walk_forward(data, forecast_funct=combination_forecast, **long_args),
}

print(compare_models(results, horizon=60))          # NOTE horizon=60, otherwise the P&L metrics are wrong
for loss in ["ae", "se"]:                           # ae -> compare with MAE, se -> compare with RMSE
    print(f"\nDM test vs Random Walk, loss = {loss}")
    print(dm_table(results, benchmark="Random Walk", horizon=60, loss=loss).round(4))

# GARCH interval vs constant-volatility interval, evaluated out of sample
garch_res = walk_forward_interval(data, interval_fn=garch_interval_forecast, **long_args)
const_res = walk_forward_interval(data, interval_fn=garch_interval_forecast, vol="constant", **long_args)
print(pd.DataFrame({"GARCH": interval_metrics(garch_res), "Constant vol": interval_metrics(const_res)}))

# Final forecast at 2026-09-30 with a GARCH 95% interval around the lag_1/lag_60 point forecast
origin_idx = data.index.searchsorted(pd.Timestamp("2026-09-30"), side="right") - 1
final = garch_interval_forecast(data, origin_idx, 60, origin_idx, lookback=3000,
                                center_fn=partial(direct_lags_forecast, short=1, long=60, lookback=3000))
print(final)

                           MAE       RMSE       MAPE  directional_accuracy  \
model                                                                        
Random Walk          14.490716  22.169808  16.181882              0.000000   
Direct lag_1 lag_60  12.156660  19.564622  13.258065              0.736018   
Direct forecast      12.246353  19.712169  13.330732              0.740492   
Combination          12.697711  20.098201  13.930642              0.736018   

                     precision_up  recall_up     f1_up  precision_down  \
model                                                                    
Random Walk              0.000000   0.000000  0.000000        0.000000   
Direct lag_1 lag_60      0.758242   0.650943  0.700508        0.720755   
Direct forecast          0.763736   0.655660  0.705584        0.724528   
Combination              0.755435   0.655660  0.702020        0.722433   

                     recall_down   f1_down  annualized_return  sharpe_ratio  \
model  

In [20]:
dm_table(test_results, benchmark="Random Walk", horizon=60, loss="ape")

,DM_stat,p_value,n
Historical Mean,0.201235,0.840606,447.0
AR,0.197749,0.843332,447.0
Direct forecast,-1.519252,0.129408,447.0
Direct lag_1 lag_60,-1.499308,0.134501,447.0
Direct lag_5 lag_60,-1.437505,0.151276,447.0
Mean Revert,-0.551172,0.581792,447.0
XGBoost,1.018411,0.309035,447.0
Direct ridge,-1.787502,0.074535,447.0


### ==== Final 60-day forecast =====
origin = 2026-09-30

In [19]:
ORIGIN_DATE = "2026-09-30"
HORIZON, LOOKBACK = 60, 3000        # 60 trading days ahead, 3000-day fitting window
SHORT, LONG = 1, 60                 # r_1d and lagged r_60d (non-overlapping, days t-59..t-1)

# 1) Locate the origin (last available trading day on or before ORIGIN_DATE)
dates = data.index
origin_idx = dates.searchsorted(pd.Timestamp(ORIGIN_DATE), side="right") - 1
origin_date = dates[origin_idx]
if origin_date != pd.Timestamp(ORIGIN_DATE):
    print(f"WARNING: {ORIGIN_DATE} not in data, using {origin_date.date()} instead")
fit_idx = origin_idx                # model fitted only on targets known by the origin
start = fit_idx - LOOKBACK + 1
assert start >= 0, "not enough history"

# 2) Rebuild the training set exactly as direct_lags_forecast does (so we can inspect the fit)
lp = np.log(data["brent_spot"].to_numpy(float))        # log of PRICE (not of log_return)

def make_X(t):
    return pd.DataFrame({
        "r_1d":         lp[t] - lp[t - SHORT],         # latest 1-day log return
        "r_60d_lagged": lp[t - SHORT] - lp[t - LONG],  # log return over t-59..t-1
    })

t = np.arange(start + LONG, fit_idx - HORIZON + 1)     # last target = fit_idx
X = make_X(t)
y = pd.Series(lp[t + HORIZON] - lp[t], name="ret_60d_ahead")

fit = sm.OLS(y.values, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": HORIZON})

# 3) Parameters and significance (Newey-West/HAC, because 60-day targets overlap)
print(fit.summary())
ci = np.asarray(fit.conf_int())
names = fit.model.exog_names
params = pd.DataFrame({"coef": np.asarray(fit.params), "HAC se": np.asarray(fit.bse),
                       "t": np.asarray(fit.tvalues), "p-value": np.asarray(fit.pvalues),
                       "95% low": ci[:, 0], "95% high": ci[:, 1]}, index=names)
print("\n", params.round(4))
print(f"\nobs = {len(y)}  (~{len(y)//HORIZON} non-overlapping 60-day windows)")
print(f"R2 = {fit.rsquared:.4f}   joint Wald p-value = {fit.f_pvalue:.4f}")

# 4) Forecast from the origin
x0 = np.r_[1.0, make_X(np.array([origin_idx])).to_numpy()[0]]
beta = np.asarray(fit.params)
pred_ret = float(x0 @ beta)
se_mean = float(np.sqrt(x0 @ np.asarray(fit.cov_params()) @ x0))   # uncertainty of expected return
sigma = float(np.sqrt(fit.scale))                       # residual sd of a 60-day return
sd_total = np.sqrt(se_mean**2 + sigma**2)
p0 = data["brent_spot"].iloc[origin_idx]
f = lambda r: p0 * np.exp(r)

# sanity check against the function used in the backtest
fc_fn = direct_lags_forecast(data, origin_idx, horizon=HORIZON, fit_idx=fit_idx,
                             lookback=LOOKBACK, short=SHORT, long=LONG,
                             nonoverlap=True, use_intercept=True)
assert np.isclose(f(pred_ret), fc_fn), "cell and model function disagree"

target_date = pd.bdate_range(origin_date, periods=HORIZON + 1)[-1]   # approx: ignores holidays
print(f"\nOrigin {origin_date.date()}  price = {p0:.2f}")
print(f"Features: r_1d = {x0[1]:+.4f}, r_60d_lagged = {x0[2]:+.4f}")
print(f"Predicted 60-day log return = {pred_ret:+.4f}  (expected-return 95% CI "
      f"{pred_ret-1.96*se_mean:+.4f} to {pred_ret+1.96*se_mean:+.4f})")
print(f"Point forecast for ~{target_date.date()}: {f(pred_ret):.2f}")
print(f"95% prediction interval: {f(pred_ret-1.96*sd_total):.2f} to {f(pred_ret+1.96*sd_total):.2f}")
print(f"Random-walk forecast: {p0:.2f}")

# 5) Optional: how the forecast is doing so far, if data already runs past the origin
elapsed = len(data) - 1 - origin_idx
if elapsed > 0:
    print(f"\n{elapsed} trading days elapsed. Latest {data.index[-1].date()}: "
          f"{data['brent_spot'].iloc[-1]:.2f} (realised log return {lp[-1]-lp[origin_idx]:+.4f})")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.121
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     7.035
Date:                Thu, 29 Oct 2026   Prob (F-statistic):           0.000896
Time:                        18:18:55   Log-Likelihood:                 166.33
No. Observations:                2880   AIC:                            -326.7
Df Residuals:                    2877   BIC:                            -308.8
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0144      0.027      0.533   